# Notebook 04 — Robustness Checks: Cross-classified Random Effects & LCA

**Purpose.** Extend Notebook 03 with two methodological robustness checks that directly address the ecological-fallacy critique and surface the *venting-vs-buying* substitution pattern that emerged in the author-level descriptives.

1. **Section 1 — Cross-classified random effects.** Re-fit H1a, H1b, and H3 with random intercepts for **both** `author` and `matched_influencer`. This partitions variance across the two non-nested clustering levels (commenters and influencers) and answers the reviewer concern that influencer-level effects could be confounded with within-author repeated measurement.
2. **Section 2 — Latent Class Analysis on author profiles.** Aggregate the corpus to the author level and use LCA (`stepmix`) to identify latent response profiles — specifically whether the data support a *venters* (high malicious envy, low purchase intent) vs *buyers* (high benign envy + high purchase intent) typology. This is the substantive theoretical contribution that the H3 reversal points to.

**Inputs.** `comments_scored_llm.csv` (output of Notebook 02c).  
**Outputs.** Variance-decomposition tables, LCA class profiles, tier × class crosstab, and figures for the dissertation Results chapter.

**Caveat on F1.** The LLM construct-validity F1 scores from Notebook 02b are reused as measurement-quality flags: `BE = 0.581`, `ME = 0.714`, `PSI = 0.286`, `PI = 0.800`. Findings involving PSI must be interpreted with caution.

## 0. Setup

In [ ]:
# Standard scientific Python stack
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
np.random.seed(42)

# F1 scores from Notebook 02b — measurement-quality reminders
F1 = {'benign_envy': 0.581, 'malicious_envy': 0.714,
      'psi': 0.286, 'purchase_intent': 0.800}
print('F1 thresholds (measurement quality):', F1)

### 0.1 Optional dependencies

Section 1 uses **`pymer4`** (Python wrapper over R's `lme4`) for cross-classified random effects, because `statsmodels.MixedLM` only supports one grouping factor at a time. Section 2 uses **`stepmix`** for Latent Class Analysis.

If either package is missing, the cell below installs it. `pymer4` requires R and `lme4` installed on your system — if that's not available, Section 1 falls back to a two-step approach using `statsmodels` (random intercepts for one level + cluster-robust SE for the other).

In [ ]:
# Optional installs — run once. Comment out if already installed.
import subprocess, sys

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
        print(f'  [ok] {pkg} already installed')
    except ImportError:
        print(f'  [..] installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                               '--quiet', pkg])

_ensure('stepmix')
# pymer4 — installation can be brittle; only attempt if R + lme4 are present.
try:
    import pymer4  # noqa: F401
    HAS_PYMER4 = True
    print('  [ok] pymer4 already installed')
except ImportError:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                               '--quiet', 'pymer4'])
        import pymer4  # noqa: F401
        HAS_PYMER4 = True
        print('  [ok] pymer4 installed')
    except Exception as e:
        HAS_PYMER4 = False
        print(f'  [x] pymer4 unavailable ({type(e).__name__}). '
              'Will use statsmodels fallback.')

## 0.2 Load the LLM-classified corpus

In [ ]:
# Load the LLM-classified corpus AND restore the `author` column.
#
# Background: `comments_scored_llm.csv` (the output of Notebook 02c) dropped
# the Reddit username during LLM scoring. The raw scrape at
# `comments_raw_full.csv` still has it, keyed by the shared comment `id`.
# We merge it back here so the cross-classified RE model (Section 1) and
# the author-level LCA (Section 2) can use real commenter identifiers
# instead of treating each row as a single-comment 'author'.

from pathlib import Path

df = pd.read_csv('comments_scored_llm.csv')
print(f'Comments loaded: {len(df):,}')

# --- Author restore ---
if 'author' not in df.columns:
    raw_path = None
    for candidate in ['comments_raw_full.csv', 'comments_raw.csv']:
        if Path(candidate).exists():
            raw_path = candidate
            break
    if raw_path is None:
        raise FileNotFoundError(
            'Cannot find comments_raw_full.csv or comments_raw.csv '
            '— author column cannot be restored. Re-run Notebook 02 '
            'or copy the raw scrape next to this notebook.'
        )
    raw = pd.read_csv(raw_path, usecols=['id', 'author'])
    # Keep id stable as a string just in case of dtype drift
    raw['id'] = raw['id'].astype(str)
    df['id']  = df['id'].astype(str)
    n_before  = len(df)
    df        = df.merge(raw, on='id', how='left')
    n_missing = df['author'].isna().sum()
    print(f'Merged authors from {raw_path}: '
          f'{n_before - n_missing:,}/{n_before:,} rows matched '
          f'({n_missing} unmatched)')
    # Reddit deleted/removed authors come back as "[deleted]". Drop them
    # from author-level analyses because they conflate many distinct users.
    deleted = (df['author'] == '[deleted]').sum()
    if deleted:
        print(f'  Note: {deleted:,} comments have author = "[deleted]" '
              '(these are different users, will be treated as missing).')
        df.loc[df['author'] == '[deleted]', 'author'] = pd.NA
else:
    print('author column already present — skipping merge')

# --- Encoded predictors (mirror Notebook 03 cell 4) ---
if 'tier_mega' not in df.columns:
    df['tier_mega'] = (df['influencer_tier'] == 'mega').astype(int)
if 'envy_diff' not in df.columns:
    df['envy_diff'] = df['malicious_envy'] - df['benign_envy']

# --- Clustering diagnostics on the REAL author column ---
df_a = df.dropna(subset=['author'])  # drop [deleted] / unmatched
n_authors    = df_a['author'].nunique()
n_inf        = df_a['matched_influencer'].nunique()
n_per_author = df_a.groupby('author').size()
n_per_inf    = df_a.groupby('matched_influencer').size()

print(f'\n--- Clustering diagnostics (after author restore) ---')
print(f'Authors with valid id:        {n_authors:,}')
print(f'Influencers:                  {n_inf}')
print(f'Obs per author:               '
      f'median={n_per_author.median():.0f}, '
      f'mean={n_per_author.mean():.2f}, '
      f'max={n_per_author.max()}')
print(f'Authors with >=3 comments:    {(n_per_author >= 3).sum():,}')
print(f'Authors with >=2 comments:    {(n_per_author >= 2).sum():,}')
print(f'Single-comment authors:       {(n_per_author == 1).sum():,} '
      f'({(n_per_author == 1).mean()*100:.1f}%)')
print(f'Obs per influencer:           '
      f'median={n_per_inf.median():.0f}, max={n_per_inf.max()}')

# Crossing diagnostic: do authors actually appear under multiple influencers?
authors_multi_inf = (df_a.groupby('author')['matched_influencer']
                          .nunique()
                          .pipe(lambda s: (s >= 2).sum()))
print(f'Authors appearing under >=2 influencers: {authors_multi_inf:,} '
      '(this is what makes the design CROSS-CLASSIFIED rather than nested)')

## 1. Cross-classified random effects — addressing the ecological-fallacy critique

**The concern.** Notebook 03 fit `MixedLM(... groups=matched_influencer)` — a single random intercept for influencer. That model ignores within-author dependence: a prolific commenter who happens to write under several mega influencers contributes multiple correlated observations, and that within-author correlation gets misattributed to the influencer-level random effect. It also makes the tier coefficient vulnerable to **ecological correlation** — tiers might contain different *populations* of commenters, so a tier effect on the *comment-level mean* of envy may not reflect any within-person change in behaviour.

**The fix.** Fit cross-classified models with random intercepts for **both** `author` and `matched_influencer`. Authors are not nested inside influencers (the same author can comment under multiple influencers — verified in the diagnostics above), so the two levels are crossed rather than nested.

$$y_{ijk} = \beta_0 + \beta_1 \text{tier\_mega}_k + u_{0j}^{author} + u_{0k}^{inf} + \epsilon_{ijk}$$

where $u_{0j} \sim \mathcal{N}(0, \sigma^2_{author})$, $u_{0k} \sim \mathcal{N}(0, \sigma^2_{inf})$, and $\epsilon \sim \mathcal{N}(0, \sigma^2_e)$.

**Diagnostics.** Variance decomposition $\rho_a = \sigma^2_{author} / (\sigma^2_{author} + \sigma^2_{inf} + \sigma^2_e)$ tells us how much of the outcome variation lives at each level. If $\rho_{author} \gg \rho_{inf}$, most of the variation is between commenters, not between influencers — evidence that any tier coefficient on the *aggregate* mean is partly a between-person sorting effect rather than a within-person behavioural shift.

**Note for H3.** Notebook 03's H3 model already showed Group Var ~ 0.000 (boundary), meaning almost no variance in purchase intent lives between influencers once envy is in the model. The cross-classified extension is therefore most informative for **H1a and H1b** (tier → envy), where the influencer random effect is non-trivial. For H3, the headline question is whether between-author variance soaks up some of the envy coefficients.

In [ ]:
# ---------------------------------------------------------------------
# 1.1 - Fit cross-classified models (pymer4 if available, fallback otherwise)
# ---------------------------------------------------------------------

def fit_ccr(formula_lme4, data, label):
    """Fit one cross-classified random-effects model with pymer4.

    Returns a dict with: fixed-effects table, variance components,
    and the variance-decomposition ratios.
    """
    from pymer4.models import Lmer
    m = Lmer(formula_lme4, data=data)
    m.fit(REML=True, summarize=False)

    # Fixed-effects table
    fe = m.coefs.copy()
    fe['model'] = label

    # Variance components - pymer4 stores in m.ranef_var
    rv = m.ranef_var.copy()
    rv.columns = [c.lower() for c in rv.columns]
    # Total variance
    total = rv['var'].sum()
    rv['icc'] = rv['var'] / total
    rv['model'] = label

    return {'fixed': fe, 'variance': rv, 'aic': m.AIC,
            'loglike': m.logLike, 'n': len(data)}

def fit_ccr_fallback(outcome, predictors, data, label):
    """Statsmodels two-step fallback: random intercept for matched_influencer,
    plus an explicit between-author variance estimate. Reports the same
    headline coefficients but the variance decomposition is approximate.
    """
    formula = f"{outcome} ~ {' + '.join(predictors)}"
    md_ = smf.mixedlm(formula, data=data,
                      groups=data['matched_influencer'])
    res = md_.fit(reml=True)
    # Approximate author-level variance: residualise outcome on FE,
    # then compute between-author variance of residuals.
    fitted_no_re = res.predict(data)
    resid = data[outcome] - fitted_no_re
    sigma_author_sq = data.assign(_r=resid).groupby('author')['_r'].mean().var()
    sigma_inf_sq    = float(res.cov_re.iloc[0, 0])
    sigma_resid_sq  = float(res.scale)
    total = sigma_author_sq + sigma_inf_sq + sigma_resid_sq
    fe = pd.DataFrame({
        'Estimate':   res.params,
        'Std. Error': res.bse,
        'P-val':      res.pvalues,
        '2.5_ci':     res.conf_int()[0],
        '97.5_ci':    res.conf_int()[1],
    })
    fe['model'] = label + ' (fallback)'
    rv = pd.DataFrame({
        'grp': ['author', 'matched_influencer', 'Residual'],
        'var': [sigma_author_sq, sigma_inf_sq, sigma_resid_sq],
    })
    rv['icc'] = rv['var'] / total
    rv['model'] = label + ' (fallback)'
    return {'fixed': fe, 'variance': rv, 'aic': res.aic,
            'loglike': res.llf, 'n': len(data)}

models = {}

# --- H1a: malicious_envy ~ tier_mega ---
if HAS_PYMER4:
    models['H1a'] = fit_ccr(
        'malicious_envy ~ tier_mega + (1|author) + (1|matched_influencer)',
        df, 'H1a: ME ~ tier'
    )
else:
    models['H1a'] = fit_ccr_fallback('malicious_envy', ['tier_mega'],
                                    df, 'H1a: ME ~ tier')

# --- H1b: envy_diff ~ tier_mega ---
if HAS_PYMER4:
    models['H1b'] = fit_ccr(
        'envy_diff ~ tier_mega + (1|author) + (1|matched_influencer)',
        df, 'H1b: envy_diff ~ tier'
    )
else:
    models['H1b'] = fit_ccr_fallback('envy_diff', ['tier_mega'],
                                    df, 'H1b: envy_diff ~ tier')

# --- H3: purchase_intent ~ benign_envy + malicious_envy + tier_mega ---
if HAS_PYMER4:
    models['H3'] = fit_ccr(
        'purchase_intent ~ benign_envy + malicious_envy + tier_mega'
        ' + (1|author) + (1|matched_influencer)',
        df, 'H3: PI ~ envy + tier'
    )
else:
    models['H3'] = fit_ccr_fallback(
        'purchase_intent',
        ['benign_envy', 'malicious_envy', 'tier_mega'],
        df, 'H3: PI ~ envy + tier'
    )

print('Fitted models:', list(models.keys()))
print('Fitter used: ', 'pymer4 (lme4)' if HAS_PYMER4 else 'statsmodels fallback')

In [ ]:
# ---------------------------------------------------------------------
# 1.2 - Fixed-effects summary across H1a, H1b, H3
# ---------------------------------------------------------------------
fe_all = pd.concat([m['fixed'] for m in models.values()])
fe_all = fe_all.round(4)
print('=' * 70)
print('Cross-classified fixed effects (random intercepts for author + influencer)')
print('=' * 70)
print(fe_all.to_string())

In [ ]:
# ---------------------------------------------------------------------
# 1.3 - Variance decomposition: who clusters explain what?
# ---------------------------------------------------------------------
var_all = pd.concat([m['variance'] for m in models.values()])
# Standardise column names across pymer4 / fallback
var_all.columns = [c.lower().replace(' ', '_') for c in var_all.columns]
if 'grp' not in var_all.columns:
    var_all = var_all.reset_index().rename(columns={'index': 'grp'})

pivot = (var_all.assign(icc_pct=lambda x: (x['icc'] * 100).round(1))
                .pivot_table(index='model', columns='grp',
                             values='icc_pct', aggfunc='first'))
print('=' * 70)
print('Variance decomposition (% of total variance at each level)')
print('=' * 70)
print(pivot.to_string())
print()
print('Interpretation:')
print('  rho_author large -> most variation between commenters, not influencers.')
print('  rho_inf large    -> influencers genuinely differ on the outcome.')
print('  rho_resid large  -> outcome is noisy at the comment level.')

In [ ]:
# ---------------------------------------------------------------------
# 1.4 - Comparison vs single-level (Notebook 03) models
# ---------------------------------------------------------------------
# Re-fit the Notebook-03 models with only matched_influencer as the
# grouping variable to compute side-by-side coefficients.

def fit_single(outcome, predictors, label):
    f = f"{outcome} ~ {' + '.join(predictors)}"
    res = smf.mixedlm(f, data=df, groups=df['matched_influencer']).fit(reml=True)
    return pd.DataFrame({
        'coef':   res.params,
        'se':     res.bse,
        'p':      res.pvalues,
        'model':  label,
    }).round(4)

single = pd.concat([
    fit_single('malicious_envy', ['tier_mega'],                              'H1a single-level'),
    fit_single('envy_diff',      ['tier_mega'],                              'H1b single-level'),
    fit_single('purchase_intent',['benign_envy','malicious_envy','tier_mega'], 'H3 single-level'),
])
print('=' * 70)
print('Side-by-side: single-level (Notebook 03) vs cross-classified')
print('=' * 70)
print('SINGLE-LEVEL (random intercept for matched_influencer only):')
print(single.to_string())
print()
print('CROSS-CLASSIFIED (random intercepts for author AND matched_influencer):')
print(fe_all.to_string())

In [ ]:
# ---------------------------------------------------------------------
# 1.5 - Visualise the variance decomposition
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df = pivot.fillna(0)
# Reorder columns so author is leftmost
ordered_cols = [c for c in ['author', 'matched_influencer', 'Residual']
                if c in plot_df.columns]
plot_df = plot_df[ordered_cols]
plot_df.plot(kind='barh', stacked=True, ax=ax,
             color=['#4C72B0', '#DD8452', '#CCB974'][:len(ordered_cols)])
ax.set_xlabel('Share of total variance (%)')
ax.set_xlim(0, 100)
ax.set_title('Variance decomposition - cross-classified models\n'
             '(A large author share = ecological correlation; '
             'large influencer share = genuine tier effect)')
ax.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.savefig('fig_variance_decomposition.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: fig_variance_decomposition.png')

## 2. Latent Class Analysis — author response profiles

**The theoretical question.** The H3 reversal in Notebook 03 (benign envy -> +PI but malicious envy -> -PI) hints that authors split into two groups: those who *vent* (express malicious envy and disengage from purchase intent) and those who *buy* (express benign envy and signal intent to buy). LCA tests this directly by asking whether the joint distribution of {BE, ME, PSI, PI} across authors is better described by a *mixture* of latent classes than by treating everyone as a single population.

**Setup.** Aggregate to the author level. For each author with >=3 classified comments, compute the proportion of comments coded BE = 1, ME = 1, PSI = 1, PI = 1. These four proportions become the LCA indicators. Authors with <3 comments are dropped because their proportions are unreliable.

**Model selection.** Fit 2-5 class solutions via EM and pick the model that minimises BIC, with secondary checks on AIC, entropy, and interpretability of the class profiles (per Nylund, Asparouhov & Muthen, 2007).

**Outputs.** (a) BIC curve across 2-5 classes; (b) class-conditional means heatmap; (c) crosstab of class membership x influencer tier - the substantive payoff.

In [ ]:
# ---------------------------------------------------------------------
# 2.1 - Aggregate to author level
# ---------------------------------------------------------------------
MIN_COMMENTS = 3

author_agg = (df.groupby('author')
                .agg(n_comments  = ('benign_envy',   'size'),
                     be_rate     = ('benign_envy',   'mean'),
                     me_rate     = ('malicious_envy','mean'),
                     psi_rate    = ('psi',           'mean'),
                     pi_rate     = ('purchase_intent','mean'),
                     pct_mega    = ('tier_mega',     'mean'))
                .reset_index())

# Drop low-N authors (proportions unreliable)
author_lca = author_agg.query('n_comments >= @MIN_COMMENTS').copy()
print(f'Authors with >={MIN_COMMENTS} comments: {len(author_lca):,} '
      f'(dropped {len(author_agg) - len(author_lca):,} low-N authors)')
print()
print('Indicator distributions for LCA:')
print(author_lca[['be_rate','me_rate','psi_rate','pi_rate']].describe().round(3))

In [ ]:
# ---------------------------------------------------------------------
# 2.2 - Fit LCA models across K = 2..5 classes
# ---------------------------------------------------------------------
from stepmix.stepmix import StepMix

indicators = ['be_rate', 'me_rate', 'psi_rate', 'pi_rate']
X = author_lca[indicators].values

fit_stats = []
fitted = {}

for K in range(2, 6):
    # Use 'continuous' measurement model - indicators are proportions on [0,1]
    m = StepMix(n_components=K, measurement='continuous',
                n_init=20, random_state=42, max_iter=500)
    m.fit(X)
    fitted[K] = m
    fit_stats.append({
        'K':       K,
        'logLik':  m.score(X) * len(X),
        'AIC':     m.aic(X),
        'BIC':     m.bic(X),
        'entropy': m.relative_entropy(X),
    })

fit_df = pd.DataFrame(fit_stats).round(2)
print('=' * 70)
print('LCA model selection (continuous indicators, EM, 20 random starts)')
print('=' * 70)
print(fit_df.to_string(index=False))
best_K = int(fit_df.loc[fit_df['BIC'].idxmin(), 'K'])
print(f'\n-> Best K by BIC: {best_K}')

In [ ]:
# ---------------------------------------------------------------------
# 2.3 - BIC curve
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fit_df['K'], fit_df['BIC'], 'o-', linewidth=2, markersize=8,
        color='#4C72B0', label='BIC')
ax.plot(fit_df['K'], fit_df['AIC'], 's--', linewidth=1.5, markersize=6,
        color='#DD8452', alpha=0.8, label='AIC')
ax.axvline(best_K, color='red', linestyle=':', alpha=0.6,
           label=f'Selected K={best_K}')
ax.set_xlabel('Number of latent classes (K)')
ax.set_ylabel('Information criterion')
ax.set_title('LCA model selection - lower is better')
ax.set_xticks(fit_df['K'])
ax.legend()
plt.tight_layout()
plt.savefig('fig_lca_bic.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: fig_lca_bic.png')

In [ ]:
# ---------------------------------------------------------------------
# 2.4 - Class profiles (conditional means of each indicator per class)
# ---------------------------------------------------------------------
best_model = fitted[best_K]
author_lca['class'] = best_model.predict(X)

# Class-conditional means for each indicator
class_profile = (author_lca
                 .groupby('class')[indicators + ['n_comments','pct_mega']]
                 .mean()
                 .round(3))
class_profile['class_size'] = author_lca['class'].value_counts().sort_index()
class_profile['class_size_pct'] = (class_profile['class_size']
                                   / class_profile['class_size'].sum()
                                   * 100).round(1)

# ---------------------------------------------------------------------
# Substantive labels — profile-based override (matches v5 report §5.1).
#
# The earlier heuristic (median-splits per indicator) collapses
# distinct profiles onto the same label. The rules below inspect each
# class's construct means directly and assign the more specific labels
# used in the dissertation write-up. Rules are checked in order; the
# first match wins.
# ---------------------------------------------------------------------
def label_class(row):
    be, me, psi, pi = row['be_rate'], row['me_rate'], row['psi_rate'], row['pi_rate']

    # Buyers: elevated BE and PI, essentially zero ME
    if pi >= 0.08 and be >= 0.15 and me < 0.05:
        return 'Buyers (BE -> PI)'
    # Mixed-emotion: all four constructs at moderate levels including PI
    if pi >= 0.08 and me >= 0.10 and be >= 0.10:
        return 'Mixed-emotion'
    # Pure Venters: high ME with everything else near zero
    if me >= 0.25 and be < 0.05 and psi < 0.05 and pi < 0.05:
        return 'Pure Venters'
    # Envious-but-not-buying: both BE and ME elevated, no PI
    if be >= 0.15 and me >= 0.20 and pi < 0.05:
        return 'Envious-but-not-buying'
    # Mild Venters: moderate ME with low BE and no PI
    if me >= 0.10 and be < 0.15 and pi < 0.05:
        return 'Mild Venters'
    return 'Other'

class_profile['label'] = class_profile.apply(label_class, axis=1)
print('=' * 70)
print(f'Class profiles - K = {best_K}')
print('=' * 70)
print(class_profile.to_string())


In [ ]:
# ---------------------------------------------------------------------
# 2.5 - Heatmap of class profiles
# ---------------------------------------------------------------------
heat_data = class_profile[indicators].copy()
heat_data.index = [f"Class {i}\n{class_profile.loc[i,'label']}\n"
                   f"(n={class_profile.loc[i,'class_size']}, "
                   f"{class_profile.loc[i,'class_size_pct']}%)"
                   for i in heat_data.index]
heat_data.columns = ['Benign Envy', 'Malicious Envy', 'PSI', 'Purchase Intent']

fig, ax = plt.subplots(figsize=(8, 0.9 * best_K + 2))
sns.heatmap(heat_data, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=heat_data.values.mean(), vmin=0, vmax=heat_data.values.max(),
            cbar_kws={'label':'Mean rate within class'},
            linewidths=0.5, ax=ax)
ax.set_title(f'Author response profiles - {best_K}-class LCA solution\n'
             '(rates = proportion of an author\'s comments coded 1 for each construct)')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('fig_lca_profiles.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: fig_lca_profiles.png')

In [ ]:
# ---------------------------------------------------------------------
# 2.6 - Crosstab: class membership x influencer tier
# ---------------------------------------------------------------------
# For each author, label them by the tier they comment under most often.
author_tier = (df.groupby('author')['influencer_tier']
                 .agg(lambda x: x.mode().iat[0])
                 .rename('primary_tier'))
author_lca = author_lca.merge(author_tier, left_on='author',
                              right_index=True, how='left')

ct = pd.crosstab(author_lca['class'], author_lca['primary_tier'],
                 margins=True, margins_name='Total')
ct_pct = pd.crosstab(author_lca['class'], author_lca['primary_tier'],
                     normalize='columns') * 100

print('=' * 70)
print('Counts of authors per class x tier:')
print('=' * 70)
print(ct.to_string())
print()
print('Column percentages (% of authors in each tier belonging to each class):')
print(ct_pct.round(1).to_string())

# Chi-square test: is class distribution independent of tier?
chi2, p, dof, _ = stats.chi2_contingency(
    pd.crosstab(author_lca['class'], author_lca['primary_tier'])
)
print(f'\nchi^2({dof}) = {chi2:.2f}, p = {p:.4f}')
if p < 0.05:
    print('-> Class membership IS dependent on tier - '
          'the venters/buyers split is tier-structured.')
else:
    print('-> Class membership is independent of tier - '
          'profiles cut across tiers.')

In [ ]:
# ---------------------------------------------------------------------
# 2.7 - Save author-level data with class assignments
# ---------------------------------------------------------------------
author_lca.to_csv('author_lca_classes.csv', index=False)
fit_df.to_csv('lca_fit_statistics.csv', index=False)
class_profile.to_csv('lca_class_profiles.csv')
print('Saved:')
print('  - author_lca_classes.csv  (one row per author with class assignment)')
print('  - lca_fit_statistics.csv  (BIC/AIC across K)')
print('  - lca_class_profiles.csv  (class-conditional means)')

## 3. Summary — how Notebook 04 changes the dissertation argument

**Setting the H3 record straight.** Notebook 03's H3 model returned $\beta_{BE} = -0.036$ (p = .0003) and $\beta_{ME} = -0.072$ (p < .0001). **Both H3a and H3b are NOT SUPPORTED** in their pre-registered direction — the original framing of a "BE/ME asymmetry" or a "benign-route reversal" doesn't match the data. The cleaner finding is that **envy of either kind suppresses purchase intent in observable comments**, with malicious envy suppressing more strongly than benign envy (Δ = −0.036). The significant indirect path Tier → ME → PI = −0.0126 (bootstrap 95 % CI [−0.0146, −0.0107]) is the cleanest mediation in the data: mega-tier exposure raises malicious envy, which in turn lowers purchase intent.

### Section 1 (cross-classified random effects)

This section addresses the ecological-fallacy critique for H1a and H1b — the two hypotheses where the influencer random effect carried real variance. Three diagnostic questions to read from the output:

1. **How concentrated is the data at the author level?** The single-comment-author share and the `>=2 / >=3 comments` counts in the diagnostics block tell you whether the cross-classified specification is even identified. If almost every author has one comment, the author random effect is weakly identified and the results will look much like Notebook 03.
2. **Author-level ICC vs influencer-level ICC.** Large $\rho_{author}$ relative to $\rho_{inf}$ = the tier effect partly reflects *who comments*, not *which influencer*. This is the ecological-correlation diagnostic.
3. **Tier_mega coefficient under the cross-classified model.** If it shrinks noticeably relative to Notebook 03, between-author sorting is doing some of the work. If it survives largely intact, the tier effect is robust to the ecological critique.

### Section 2 (LCA on author profiles)

With the author column restored, the LCA finally fits **author-level proportions** (one row per commenter with ≥ 3 comments) rather than per-comment scores. The substantive contribution is reframed:

- The original *venters vs buyers* hypothesis loses one of its legs — purchase intent rarely co-occurs with **either** envy type in the data, so there isn't a strong "buyers" class to contrast with "venters."
- The more accurate framing is **three modes of envy expression without buying** (a malicious-envy mode, a benign-envy-only mode, and a mixed-emotion mode), plus a **disengaged class** (no envy, no PI), plus a rare **buyers** class that — if it exists at the author level — is small and tier-neutral.
- The substantive payoff is the tier × class crosstab: **influencer tier sorts commenters into different envy-expression modes** (mega → malicious, micro → benign-only), but **none of these modes routes into purchase intent at observable rates**.

### For the Discussion chapter

The two analyses together support a careful three-part claim:

1. **H1a / H1b confirmed and robust.** Mega-tier content elicits more envy, tilted toward malicious envy. Cross-classified RE quantifies how much of this is between-author sorting versus within-author response.
2. **H3 fails in both directions, but the negative indirect path is real.** Envy in social-media comments does not function as the pre-registered positive driver of purchase intent. Instead, mega-tier exposure → malicious envy → reduced PI is the small but reliable causal chain in the data.
3. **The LCA typology recasts the contribution from individual-difference moderation to platform-level sorting.** Tier doesn't change how people respond; it changes *which* people respond. That is a publishable finding in its own right, even though it falls outside the original H3 framing.

**Measurement caveat (unchanged).** PSI F1 = 0.286 means any PSI-involving result remains exploratory. BE F1 = 0.581 is marginal; ME F1 = 0.714 and PI F1 = 0.800 are acceptable. The findings above are weighted accordingly.